# CrashDiag base-Qwen hard evaluation on Kaggle

This independent, evaluation-only notebook measures the untrained `Qwen/Qwen2.5-1.5B-Instruct` base model against the exact signed 192-row schema-v2 hard split. It uses deterministic generation and executes each proposed JSON action in the CrashDiag environment. It does not run an agent loop, train weights, or use LLM judging.

In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys

WORKFLOW_VERSION = "base-qwen-hard-evaluation-v1"
REPO_URL = "https://github.com/Indium-AI-Labs/CrashDiag.git"
REPO_DIR = Path("/kaggle/working/CrashDiag")
BUCKET_ID = "devaanshpa/CrashDiag"
SANDBOX_URL = "https://sandbox.devaanshpathak.com"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
HARD_RUN_ID = os.environ.get("CRASHDIAG_HARD_RUN_ID") or "20260720T164228Z-grpo-hard-7aa31d7f3710"
HARD_SOURCE_COMMIT = os.environ.get("CRASHDIAG_HARD_SOURCE_COMMIT") or "f732e5fed815a73a53c8ee860c3fd865a6577fb2"
EVALUATOR_COMMIT = os.environ.get("CRASHDIAG_EVALUATOR_COMMIT") or "596bda193bb3101c59c9e454997b36d617b4b000"
BASE_QWEN_RUN_ID = os.environ.get("CRASHDIAG_BASE_QWEN_RUN_ID") or "base-qwen-hard-7aa31d7f3710"
BASE_QWEN_STAGE = "base-qwen-hard-evaluation"
PRECISION = "auto"
EXPECTED_ROWS = 192

for name, value in (("HARD_SOURCE_COMMIT", HARD_SOURCE_COMMIT), ("EVALUATOR_COMMIT", EVALUATOR_COMMIT)):
    if re.fullmatch(r"[0-9a-f]{40}", value) is None:
        raise ValueError(f"{name} must be a full lowercase Git SHA")
print(f"WORKFLOW_VERSION={WORKFLOW_VERSION}\nHARD_RUN_ID={HARD_RUN_ID}\nBASE_QWEN_RUN_ID={BASE_QWEN_RUN_ID}\nEVALUATOR_COMMIT={EVALUATOR_COMMIT}")

## Install the pinned evaluator

In [ ]:
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"Refusing to overwrite {REPO_DIR}")
else:
    subprocess.run(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", EVALUATOR_COMMIT], check=True)
CURRENT_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if CURRENT_COMMIT != EVALUATOR_COMMIT:
    raise RuntimeError("evaluator checkout mismatch")
torchao_probe = subprocess.run([sys.executable, "-m", "pip", "show", "torchao"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
if torchao_probe.returncode == 0:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train]"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"checked_out_evaluator={CURRENT_COMMIT}")

## Load Kaggle secrets

In [ ]:
def required_secret(name: str) -> str:
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception as exc:
        raise RuntimeError(f"Attach Kaggle Secret {name!r}") from exc
    if not value:
        raise RuntimeError(f"Kaggle Secret {name!r} is empty")
    return value

os.environ["HF_TOKEN"] = required_secret("HF_TOKEN")
os.environ["CRASHDIAG_SANDBOX_TOKEN"] = required_secret("CRASHDIAG_SANDBOX_TOKEN")
os.environ["CRASHDIAG_SANDBOX_URL"] = SANDBOX_URL
os.environ["CRASHDIAG_HF_BUCKET_ID"] = BUCKET_ID
os.environ["CRASHDIAG_ARTIFACT_PREFIX"] = "runs"
os.environ["CRASHDIAG_ARTIFACT_LOCAL_ROOT"] = str(REPO_DIR / "artifacts")
os.environ["CRASHDIAG_ARTIFACT_UPLOAD_POLICY"] = "required"
os.environ["CRASHDIAG_RUN_ID"] = BASE_QWEN_RUN_ID
print("Kaggle Secrets loaded; values were not printed.")

## Download and verify the signed hard split

In [ ]:
from training.artifacts import ArtifactConfig, ArtifactUploader
from training.calibrate_grpo import read_jsonl

def make_uploader(run_id: str) -> ArtifactUploader:
    return ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=run_id, prefix="runs", policy="required", local_root=REPO_DIR / "artifacts", token=os.environ["HF_TOKEN"]))

def fetch_stage(client: ArtifactUploader, stage: str, target: Path, paths: list[str]) -> Path:
    if target.exists() and any(target.iterdir()):
        client.verify_local_stage(target, stage, include_paths=paths)
    else:
        client.download_stage(stage, target, include_paths=paths)
    return target

def remote_path_exists(client: ArtifactUploader, path: str) -> bool:
    return any(getattr(item, "path", None) == path for item in client.api.get_bucket_paths_info(client.config.bucket_id, [path]))

HANDOFF = Path("/kaggle/working/crashdiag-base-qwen-hard")
hard_client = make_uploader(HARD_RUN_ID)
HARD_DATA = fetch_stage(hard_client, "datasets", HANDOFF / HARD_RUN_ID / "datasets", ["grpo_hard_eval.jsonl", "grpo_hard_summary.json"])
hard_manifest = json.loads((HARD_DATA / "manifest.json").read_text(encoding="utf-8"))
if hard_manifest.get("runtime", {}).get("git_commit") != HARD_SOURCE_COMMIT:
    raise RuntimeError("hard dataset/source commit mismatch")
hard_summary = json.loads((HARD_DATA / "grpo_hard_summary.json").read_text(encoding="utf-8"))
if hard_summary.get("curriculum_version") != 2 or hard_summary.get("action_contract") != "parameter_free_repairs":
    raise RuntimeError("hard dataset is not curriculum-v2 parameter-free data")
HARD_EVAL_FILE = HARD_DATA / "grpo_hard_eval.jsonl"
if len(read_jsonl(HARD_EVAL_FILE)) != EXPECTED_ROWS:
    raise RuntimeError("hard evaluation split is not exactly 192 rows")
print(f"verified exact {EXPECTED_ROWS}-row hard split from {hard_client.remote_uri('datasets')}")

## Verify the live environment and GPU

In [ ]:
from crashdiag.sandbox_apps.http import HttpSandbox
import torch

with HttpSandbox(SANDBOX_URL, api_token=os.environ["CRASHDIAG_SANDBOX_TOKEN"], timeout=15.0) as sandbox:
    service = sandbox.service_health()
    application = sandbox.health_check()
if 2 not in service.get("scenario_schema_versions", []):
    raise RuntimeError(f"Vultr service lacks schema v2: {service}")
if service.get("hard_scenario_batch") is not True:
    raise RuntimeError("Vultr service lacks atomic hard-scenario setup")
if application.get("healthy") is not True:
    raise RuntimeError(f"sandbox preflight failed: {application}")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU")
print(f"gpu={torch.cuda.get_device_name(0)}, bf16={torch.cuda.is_bf16_supported()}")

## Evaluate the base policy and sign the result

In [ ]:
from IPython.display import SVG, display
from training.evaluate_jsonl import main as evaluate_jsonl_main

uploader = make_uploader(BASE_QWEN_RUN_ID)
BASE_QWEN_OUTPUT = REPO_DIR / "outputs/base-qwen-hard-evaluation"
BASE_QWEN_CACHE = HANDOFF / BASE_QWEN_RUN_ID / BASE_QWEN_STAGE
BASE_QWEN_PATHS = ["mechanical_evaluation.json", "reports/mechanical_evaluation_summary.json", "reports/mechanical_success_by_fault.svg"]
stage_complete = uploader.stage_is_complete(BASE_QWEN_STAGE)
run_complete = remote_path_exists(uploader, f"{uploader.config.remote_root}/_SUCCESS.json")
if run_complete and not stage_complete:
    raise RuntimeError("base-Qwen run is complete but its evaluation stage is missing")
if not run_complete and not stage_complete:
    uploader.start_run({"workflow": WORKFLOW_VERSION, "model": BASE_MODEL, "hard_run_id": HARD_RUN_ID, "evaluator_commit": EVALUATOR_COMMIT, "scoring": "mechanical_fault_resolution"})

if stage_complete:
    BASE_QWEN_DIR = fetch_stage(uploader, BASE_QWEN_STAGE, BASE_QWEN_CACHE, BASE_QWEN_PATHS)
    print(f"reused signed stage {BASE_QWEN_STAGE}")
else:
    BASE_QWEN_DIR = BASE_QWEN_OUTPUT
    base_exit = evaluate_jsonl_main([
        "--model", BASE_MODEL,
        "--dataset", str(HARD_EVAL_FILE),
        "--output-dir", str(BASE_QWEN_DIR),
        "--precision", PRECISION,
        "--max-new-tokens", "96",
        "--artifact-stage", BASE_QWEN_STAGE,
    ])
    if base_exit != 0:
        raise RuntimeError(f"base-Qwen evaluation failed with status {base_exit}")

base_report = json.loads((BASE_QWEN_DIR / "mechanical_evaluation.json").read_text(encoding="utf-8"))
summary = base_report["summary"]
if summary.get("total_episodes") != EXPECTED_ROWS:
    raise RuntimeError("base-Qwen report did not evaluate all hard rows")
if summary.get("backend_error_rate") != 0.0:
    raise RuntimeError("base-Qwen evaluation had sandbox backend errors")
if not run_complete:
    uploader.complete_run({"stages": [BASE_QWEN_STAGE], "workflow": WORKFLOW_VERSION, "model": BASE_MODEL, "hard_run_id": HARD_RUN_ID, "evaluator_commit": EVALUATOR_COMMIT, "scoring": "mechanical_fault_resolution"})
print(json.dumps(summary, indent=2, sort_keys=True))
for chart in sorted((BASE_QWEN_DIR / "reports").glob("*.svg")):
    display(SVG(filename=str(chart)))
print(f"BASE_QWEN_ARTIFACTS={uploader.remote_uri()}")